# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [2]:
!pip install imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable


In [3]:
#Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from sklearn.utils import resample

from imblearn.over_sampling import SMOTE

In [5]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


In [6]:
# Revisar tamaño del dataset
fraud.shape

(1000000, 8)

In [7]:
# Revisar tipos de datos
fraud.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 8 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   distance_from_home              1000000 non-null  float64
 1   distance_from_last_transaction  1000000 non-null  float64
 2   ratio_to_median_purchase_price  1000000 non-null  float64
 3   repeat_retailer                 1000000 non-null  float64
 4   used_chip                       1000000 non-null  float64
 5   used_pin_number                 1000000 non-null  float64
 6   online_order                    1000000 non-null  float64
 7   fraud                           1000000 non-null  float64
dtypes: float64(8)
memory usage: 61.0 MB


In [9]:
# Revisar valores nulos
fraud.isna().sum()

distance_from_home                0
distance_from_last_transaction    0
ratio_to_median_purchase_price    0
repeat_retailer                   0
used_chip                         0
used_pin_number                   0
online_order                      0
fraud                             0
dtype: int64

**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

## Distribución de la variable objetivo

In [10]:
# Conteo de clases
fraud["fraud"].value_counts()

fraud
0.0    912597
1.0     87403
Name: count, dtype: int64

In [11]:
# Porcentaje de clases
fraud["fraud"].value_counts(normalize=True) * 100

fraud
0.0    91.2597
1.0     8.7403
Name: proportion, dtype: float64

## Distribución de la variable objetivo

La variable objetivo es `fraud`.

- 0 representa una transacción legítima.
- 1 representa una transacción fraudulenta.

Al revisar la distribución de clases, podemos comprobar si el dataset está desbalanceado.

Si la clase 0 representa una proporción mucho mayor que la clase 1, entonces estamos trabajando con un problema de clasificación desbalanceada.

## Separar X e y

In [12]:
# Definir variables predictoras y target
X = fraud.drop(columns=["fraud"])
y = fraud["fraud"]

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1000000, 7)
y shape: (1000000,)


In [13]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (800000, 7)
X_test: (200000, 7)
y_train: (800000,)
y_test: (200000,)


## Modelo base: Logistic Regression

In [14]:
# Entrenar modelo base
log_reg = LogisticRegression(max_iter=1000)

log_reg.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [15]:
# Evaluar modelo base
y_pred = log_reg.predict(X_test)
y_proba = log_reg.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.959285
ROC-AUC: 0.9670504340402107

Matriz de confusión:
[[181292   1227]
 [  6916  10565]]

Classification report:
              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98    182519
         1.0       0.90      0.60      0.72     17481

    accuracy                           0.96    200000
   macro avg       0.93      0.80      0.85    200000
weighted avg       0.96      0.96      0.96    200000



## Evaluación del modelo base

En problemas de fraude, el accuracy no es suficiente para evaluar el modelo.

La métrica más importante suele ser el recall de la clase fraudulenta, porque queremos detectar la mayor cantidad posible de fraudes reales.

Si el modelo tiene accuracy alto pero recall bajo para la clase 1, significa que está clasificando bien muchas transacciones legítimas, pero está fallando al detectar fraudes.

## Oversampling
Qué es Oversampling

Oversampling aumenta la clase minoritaria duplicando ejemplos existentes.

En este caso:

Aumentamos las transacciones fraudulentas para equilibrarlas con las legítimas.

In [16]:
# Crear dataset de entrenamiento combinado
train_data = pd.concat([X_train, y_train], axis=1)

majority_class = train_data[train_data["fraud"] == 0]
minority_class = train_data[train_data["fraud"] == 1]

print("Clase mayoritaria:", majority_class.shape)
print("Clase minoritaria:", minority_class.shape)

Clase mayoritaria: (730078, 8)
Clase minoritaria: (69922, 8)


In [17]:
# Aplicar oversampling
minority_oversampled = resample(
    minority_class,
    replace=True,
    n_samples=len(majority_class),
    random_state=42
)

train_oversampled = pd.concat([majority_class, minority_oversampled])

train_oversampled["fraud"].value_counts()

fraud
0.0    730078
1.0    730078
Name: count, dtype: int64

In [18]:
# Separar X e y oversampled
X_train_over = train_oversampled.drop(columns=["fraud"])
y_train_over = train_oversampled["fraud"]

In [19]:
# Entrenar modelo con oversampling
log_reg_over = LogisticRegression(max_iter=1000)

log_reg_over.fit(X_train_over, y_train_over)

y_pred_over = log_reg_over.predict(X_test)
y_proba_over = log_reg_over.predict_proba(X_test)[:, 1]

In [20]:
# Evaluar oversampling
print("Accuracy:", accuracy_score(y_test, y_pred_over))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_over))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_over))

print("\nClassification report:")
print(classification_report(y_test, y_pred_over))

Accuracy: 0.934805
ROC-AUC: 0.9795256160986981

Matriz de confusión:
[[170388  12131]
 [   908  16573]]

Classification report:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



## Undersampling
Qué es Undersampling

Undersampling reduce la clase mayoritaria eliminando ejemplos.

En este caso:

Reducimos las transacciones legítimas para igualarlas con las fraudulentas.

Ventaja:

Equilibra el dataset rápidamente.

Riesgo:

Puede perder mucha información útil.

In [21]:
# Aplicar undersampling
majority_undersampled = resample(
    majority_class,
    replace=False,
    n_samples=len(minority_class),
    random_state=42
)

train_undersampled = pd.concat([majority_undersampled, minority_class])

train_undersampled["fraud"].value_counts()

fraud
0.0    69922
1.0    69922
Name: count, dtype: int64

In [22]:
# Separar X e y undersampled
X_train_under = train_undersampled.drop(columns=["fraud"])
y_train_under = train_undersampled["fraud"]

In [23]:
# Entrenar modelo con undersampling
log_reg_under = LogisticRegression(max_iter=1000)

log_reg_under.fit(X_train_under, y_train_under)

y_pred_under = log_reg_under.predict(X_test)
y_proba_under = log_reg_under.predict_proba(X_test)[:, 1]

In [24]:
# Evaluar undersampling
print("Accuracy:", accuracy_score(y_test, y_pred_under))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_under))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_under))

print("\nClassification report:")
print(classification_report(y_test, y_pred_under))

Accuracy: 0.934745
ROC-AUC: 0.9795777091963628

Matriz de confusión:
[[170385  12134]
 [   917  16564]]

Classification report:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



## SMOTE
Qué es SMOTE

SMOTE significa:

Synthetic Minority Oversampling Technique

En lugar de duplicar fraudes existentes, crea ejemplos sintéticos de la clase minoritaria.

Es una técnica más sofisticada que oversampling simple.

In [25]:
# Aplicar SMOTE
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

y_train_smote.value_counts()

/Users/gabrielbohorquez/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


fraud
0.0    730078
1.0    730078
Name: count, dtype: int64

In [26]:
# Entrenar modelo con SMOTE
log_reg_smote = LogisticRegression(max_iter=1000)

log_reg_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = log_reg_smote.predict(X_test)
y_proba_smote = log_reg_smote.predict_proba(X_test)[:, 1]

In [27]:
# Evaluar SMOTE
print("Accuracy:", accuracy_score(y_test, y_pred_smote))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_smote))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred_smote))

print("\nClassification report:")
print(classification_report(y_test, y_pred_smote))

Accuracy: 0.93519
ROC-AUC: 0.9791798541923508

Matriz de confusión:
[[170497  12022]
 [   940  16541]]

Classification report:
              precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182519
         1.0       0.58      0.95      0.72     17481

    accuracy                           0.94    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.94      0.94    200000



## Comparar resultados

In [29]:
# Tabla comparativa
results = pd.DataFrame({
    "Model": [
        "Logistic Regression Base",
        "Logistic Regression Oversampling",
        "Logistic Regression Undersampling",
        "Logistic Regression SMOTE"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_over),
        accuracy_score(y_test, y_pred_under),
        accuracy_score(y_test, y_pred_smote)
    ],
    "ROC_AUC": [
        roc_auc_score(y_test, y_proba),
        roc_auc_score(y_test, y_proba_over),
        roc_auc_score(y_test, y_proba_under),
        roc_auc_score(y_test, y_proba_smote)
    ]
})

results

,Model,Accuracy,ROC_AUC
0,Logistic Regression Base,0.959285,0.967050
1,Logistic Regression Oversampling,0.934805,0.979526
2,Logistic Regression Undersampling,0.934745,0.979578
3,Logistic Regression SMOTE,0.935190,0.979180


In [31]:
# Comparar recall de fraude - clave para verlo de forma automática:
from sklearn.metrics import recall_score, precision_score, f1_score

comparison = pd.DataFrame({
    "Model": [
        "Base",
        "Oversampling",
        "Undersampling",
        "SMOTE"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_over),
        accuracy_score(y_test, y_pred_under),
        accuracy_score(y_test, y_pred_smote)
    ],
    "Precision_Fraud": [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_pred_over),
        precision_score(y_test, y_pred_under),
        precision_score(y_test, y_pred_smote)
    ],
    "Recall_Fraud": [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_pred_over),
        recall_score(y_test, y_pred_under),
        recall_score(y_test, y_pred_smote)
    ],
    "F1_Fraud": [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_over),
        f1_score(y_test, y_pred_under),
        f1_score(y_test, y_pred_smote)
    ],
    "ROC_AUC": [
        roc_auc_score(y_test, y_proba),
        roc_auc_score(y_test, y_proba_over),
        roc_auc_score(y_test, y_proba_under),
        roc_auc_score(y_test, y_proba_smote)
    ]
})

comparison.sort_values(by="Recall_Fraud", ascending=False)

,Model,Accuracy,Precision_Fraud,Recall_Fraud,F1_Fraud,ROC_AUC
1,Oversampling,0.934805,0.577376,0.948058,0.717679,0.979526
2,Undersampling,0.934745,0.577183,0.947543,0.717382,0.979578
3,SMOTE,0.935190,0.579106,0.946227,0.718487,0.979180
0,Base,0.959285,0.895946,0.604370,0.721826,0.967050


## Conclusión final

En este laboratorio se trabajó con un dataset desbalanceado de fraude en transacciones con tarjeta.

La variable objetivo fue `fraud`, donde `0` representa una transacción legítima y `1` representa una transacción fraudulenta. Al analizar la distribución del target, se observa que la clase fraudulenta es minoritaria, por lo que accuracy no es una métrica suficiente para evaluar el modelo.

Primero se entrenó un modelo base de Logistic Regression. Después se aplicaron diferentes técnicas de balanceo: Oversampling, Undersampling y SMOTE.

La comparación entre modelos debe centrarse principalmente en el recall de la clase fraudulenta, porque en detección de fraude es especialmente importante reducir los falsos negativos, es decir, evitar que transacciones fraudulentas sean clasificadas como legítimas.

El mejor modelo será aquel que logre un buen equilibrio entre recall, precision, F1-score y ROC-AUC, priorizando la detección efectiva de fraudes.